# TVB Widgets PoC — New 3D JupyterLab Widgets
## GSoC 2026 · Project #11 · The Virtual Brain

This notebook demonstrates two new widgets for the `tvb-widgets` ecosystem,
built as a proof-of-concept for GSoC 2026 Project #11.

Both widgets extend the existing `tvb-widgets` architecture — using the same
`k3d` rendering backend, `ipywidgets` control pattern, and `TVBWidget` base
class conventions — while introducing new capabilities requested in the project
brief: 3D connectivity visualisation with live filtering, and animated
timeseries painted onto a cortical surface mesh.

**Requirements:** `tvb-data`, `tvb-library`, `k3d`, `ipywidgets`, `jupyterlab`  
All included in `pyproject.toml` — install with `pip install -e .` from the
repo root.

In [ ]:
# Verify dependencies are available
import importlib, sys
required = ['k3d', 'ipywidgets', 'tvb', 'numpy']
missing = [m for m in required if importlib.util.find_spec(m) is None]
if missing:
    print(f"Missing: {missing}. Run: pip install -e . from repo root.")
else:
    print("All dependencies available")

---
## Part 1 · Connectivity3DWidget

### Visualising the Human Connectome in 3D

The structural connectivity of the brain — the pattern of white matter
fibre tracts linking cortical regions — is one of the most important inputs
to any TVB simulation. Understanding which regions are strongly connected,
and how connectivity is distributed across hemispheres, is essential before
running any model.

The existing `tvb-widgets` ecosystem provides a 2D matrix view of
connectivity. This widget adds a fully interactive 3D representation:
regions as positioned spheres at their real anatomical coordinates, connections
as weighted line segments, with live filtering by connection strength and
hemisphere.

In [ ]:
import numpy as np
from tvb.datatypes.connectivity import Connectivity
from tvbwidgets_poc import Connectivity3DWidget

conn = Connectivity.from_file()
conn.configure()

info = {
    'Regions': len(conn.region_labels),
    'Non-zero connections': int(np.count_nonzero(conn.weights)),
    'Weight range': f"[{conn.weights[conn.weights > 0].min():.4f}, {conn.weights.max():.4f}]",
    'First 5 regions': list(conn.region_labels[:5])
}
for k, v in info.items():
    print(f"  {k}: {v}")

TVB ships a 76-region parcellation of the human connectome derived from
diffusion tractography. Each of the 76 regions corresponds to a cortical
area, and the weight matrix encodes the density of white matter fibres
connecting them. We load this directly from the `tvb-data` package —
no EBRAINS account or file download required.

Out of a possible 76×76 = 5,776 region pairs, only **1,560 connections are
non-zero** — roughly 27% density. This sparse structure reflects the
anatomical reality of the connectome: long-range fibre pathways are
metabolically expensive and exist only where evolutionary pressure has
established them. The weight range reflects normalised tractography streamline
counts: values near zero indicate sparse projections, the maximum marks the
densest cortico-cortical pathway in this parcellation (typically a
homotopic interhemispheric connection).

In [ ]:
# Structured connectivity summary — available programmatically via get_connectivity_info()
w1 = Connectivity3DWidget(conn)
summary = w1.get_connectivity_info()
print(f"  Regions: {summary['n_regions']}")
print(f"  Connections: {summary['n_connections']}")
print(f"  Weight min: {summary['weight_min']:.4f}")
print(f"  Weight max: {summary['weight_max']:.4f}")

### Rendering the Connectome

The widget renders below. Each sphere is a brain region positioned at its
real MNI coordinate. Edge thickness and the colourmap reflect connection
weight. Use the controls to:

- **Threshold slider** — hide connections below a weight fraction; try `0.3`
  to reveal only the strongest structural pathways
- **Colormap** — `plasma` maps well to connection density; `coolwarm`
  emphasises the bilateral symmetry of the connectome
- **Hemisphere toggle** — isolate left or right hemisphere to inspect
  intra-hemispheric connectivity patterns

In [ ]:
display(w1)

> **Implementation note:** Connection colours are currently computed
> per brain-region vertex (one colour per node, derived from the mean
> weight of all connections incident to that node) rather than per edge.
> This is a k3d architectural constraint — `k3d.lines` with
> `indices_type='segment'` shares the vertex colour buffer rather than
> maintaining a separate per-edge colour array. A future improvement
> would switch to individual `k3d.line` objects per edge for true
> per-edge colour mapping, at the cost of some rendering performance
> for dense connectomes.

> **Hemisphere filter note:** The hemisphere toggle identifies left/right
> regions by label prefix (`l` / `r`) — a convention specific to the
> 76-region TVB default dataset. Connectivity objects with different
> naming conventions will require a custom `region_labels` mapping or
> a user-supplied hemisphere index array.

---
## Part 2 · AnimatedSurface3DWidget

### Neural Activity on a Cortical Surface

TVB simulations produce timeseries data — one signal per brain region or
per surface vertex, evolving over time. Visualising this on the cortical
surface is the most direct way to understand what a simulation is doing
spatially. Static snapshots miss the dynamics entirely.

This widget renders the full TVB cortical surface mesh (~16k vertices,
~33k triangles) and animates a timeseries array by mapping scalar values
to a diverging colourmap per vertex, per frame — with no mesh rebuild
between frames. Only the vertex attribute buffer is updated via k3d's
traitlet system, keeping the animation smooth even at 12 fps in a standard
JupyterLab environment.

In [ ]:
from tvb.datatypes.surfaces import CorticalSurface
from tvbwidgets_poc import AnimatedSurface3DWidget

surf = CorticalSurface.from_file()
surf.configure()

print(f"  Vertices:     {surf.vertices.shape}")
print(f"  Triangles:    {surf.triangles.shape}")
print(f"  Vertex dtype: {surf.vertices.dtype}")

w2 = AnimatedSurface3DWidget(surf)   # synthetic timeseries auto-generated
display(w2)

### Animation Controls

The synthetic timeseries divides the cortical surface into 8 pseudo-regions,
each oscillating at a distinct frequency (0.5 Hz to 2.6 Hz) with staggered
phase offsets — approximating the kind of spatially heterogeneous dynamics
that TVB simulations produce.

- **Play** — starts the animation (~12 fps)
- **Frame slider** — scrub to any specific frame; fully synchronised with Play
- **Speed** — 4x speed compresses 4 seconds of simulated activity into 1 second
- **Colormap** — all options are diverging (blue-white-red family), appropriate
  for signals that oscillate above and below a resting baseline

To use real TVB simulation output instead of the synthetic data, pass a
`(T, N_vertices)` float32 array at construction time.

In [ ]:
# Example: providing real simulation timeseries
# Replace this with actual TVB simulation output:
#
#   ts = your_simulation_result.astype(np.float32)  # shape: (T, N_vertices)
#   w2_real = AnimatedSurface3DWidget(surf, timeseries=ts)
#   display(w2_real)
#
# The widget auto-scales to T frames and N_vertices.
# For demonstration, generate a quick 60-frame array:
import numpy as np
N = len(surf.vertices)
ts_demo = np.random.randn(60, N).astype(np.float32) * 0.5
print(f"Demo timeseries shape: {ts_demo.shape}, dtype: {ts_demo.dtype}")
print("(Pass this as AnimatedSurface3DWidget(surf, timeseries=ts_demo) to animate real data)")

---
## Part 3 · Architecture and Extension Path

### How These Widgets Fit the tvb-widgets Ecosystem

In [ ]:
print("""
tvb-widgets-poc architecture
============================

TVBWidgetPOC (base_widget.py)
  |-- Connectivity3DWidget  (connectivity3d.py)
  |     k3d backend   : points + lines with live traitlet mutation
  |     Controls      : threshold, node size, colormap, hemisphere
  |     TVB types     : Connectivity
  |
  +-- AnimatedSurface3DWidget  (surface3d.py)
        k3d backend   : mesh with per-vertex attribute animation
        Controls      : Play/scrub, colormap, speed
        TVB types     : CorticalSurface + (T, N) timeseries

Both widgets follow the existing tvb-widgets patterns:
  * ipywidgets.VBox + TVBWidget multiple inheritance
  * k3d.plot() inside ipywidgets.Output() for JupyterLab DOM placement
  * add_datatype(HasTraits) public API
  * Defensive .configure() before accessing derived TVB attributes
""")

### Integration with tvb-ext-xircuits

The project brief requests that new widgets be integrated into the
[tvb-ext-xircuits](https://github.com/the-virtual-brain/tvb-ext-xircuits)
JupyterLab extension — a React JS-based visual programming environment.

Currently only `PhasePlaneWidget` is linked there. The integration pattern
is already established: each widget is exposed as an xircuits component that
can be wired into a visual workflow, with its `add_datatype()` method
serving as the data input port.

Both `Connectivity3DWidget` and `AnimatedSurface3DWidget` are designed with
this integration in mind:
- Single `add_datatype()` entry point — maps cleanly to an xircuits input port
- State is encapsulated — no global side effects
- `get_connectivity_info()` returns a serialisable dict — can serve as an
  xircuits output port for downstream components

A Phase 2 GSoC deliverable would wire both widgets into xircuits following
the `PhasePlaneWidget` integration as a template.

---

## Summary

| Widget | TVB datatype | Backend | Live controls |
|--------|-------------|---------|---------------|
| `Connectivity3DWidget` | `Connectivity` | k3d points + lines | Threshold, colormap, hemisphere, node size |
| `AnimatedSurface3DWidget` | `CorticalSurface` + timeseries | k3d mesh | Play/scrub, colormap, speed |

Both widgets are available at `from tvbwidgets_poc import ...` and run
locally without an EBRAINS account or `CLB_AUTH` token.

Source: [github.com/your-username/tvb-widgets-poc](https://github.com)  
Contact: [your-email@example.com]